In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 29


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 1.6233303733170033
Epoch 2/100, Loss: 1.794529341161251
Epoch 3/100, Loss: 1.9204813167452812
Epoch 4/100, Loss: 1.6438475847244263
Epoch 5/100, Loss: 1.766414813697338
Epoch 6/100, Loss: 1.8139455392956734
Epoch 7/100, Loss: 1.695086233317852
Epoch 8/100, Loss: 1.8511410057544708
Epoch 9/100, Loss: 1.8941239230334759
Epoch 10/100, Loss: 1.732495117932558
Epoch 11/100, Loss: 1.7502440102398396
Epoch 12/100, Loss: 1.855854220688343
Epoch 13/100, Loss: 2.104090176522732
Epoch 14/100, Loss: 1.6092447564005852
Epoch 15/100, Loss: 1.8245963156223297
Epoch 16/100, Loss: 2.0752538591623306
Epoch 17/100, Loss: 1.9006211273372173
Epoch 18/100, Loss: 1.5982604622840881
Epoch 19/100, Loss: 1.893095813691616


Epoch 20/100, Loss: 1.8135642781853676
Epoch 21/100, Loss: 1.8934879004955292
Epoch 22/100, Loss: 1.8886251002550125
Epoch 23/100, Loss: 1.7850342094898224
Epoch 24/100, Loss: 1.8924572467803955
Epoch 25/100, Loss: 1.7790706679224968
Epoch 26/100, Loss: 1.6689226627349854
Epoch 27/100, Loss: 1.831064447760582
Epoch 28/100, Loss: 1.8490043580532074
Epoch 29/100, Loss: 1.8123226165771484
Epoch 30/100, Loss: 1.8308930061757565
Epoch 31/100, Loss: 1.880980245769024
Epoch 32/100, Loss: 1.8386360108852386
Epoch 33/100, Loss: 1.7584707215428352
Epoch 34/100, Loss: 1.901836521923542
Epoch 35/100, Loss: 1.9455020651221275
Epoch 36/100, Loss: 1.6971843019127846
Epoch 37/100, Loss: 1.6501362770795822


Epoch 38/100, Loss: 1.9050055891275406
Epoch 39/100, Loss: 1.702194668352604
Epoch 40/100, Loss: 1.8142004013061523
Epoch 41/100, Loss: 1.6799414940178394
Epoch 42/100, Loss: 1.740301139652729
Epoch 43/100, Loss: 1.8797597736120224
Epoch 44/100, Loss: 1.8031042367219925
Epoch 45/100, Loss: 1.6484192945063114
Epoch 46/100, Loss: 1.7365874908864498
Epoch 47/100, Loss: 1.8455307632684708
Epoch 48/100, Loss: 1.7639618143439293
Epoch 49/100, Loss: 1.796413041651249
Epoch 50/100, Loss: 1.8982476890087128
Epoch 51/100, Loss: 1.8926247954368591
Epoch 52/100, Loss: 1.77288818359375
Epoch 53/100, Loss: 1.767230562865734
Epoch 54/100, Loss: 1.7809526026248932
Epoch 55/100, Loss: 1.7423739098012447


Epoch 56/100, Loss: 1.852908506989479
Epoch 57/100, Loss: 1.7958464622497559
Epoch 58/100, Loss: 1.710267223417759
Epoch 59/100, Loss: 1.6329475976526737
Epoch 60/100, Loss: 1.9454140365123749
Epoch 61/100, Loss: 1.7271243706345558
Epoch 62/100, Loss: 1.77119380235672
Epoch 63/100, Loss: 1.8646778166294098
Epoch 64/100, Loss: 1.796438328921795
Epoch 65/100, Loss: 1.7061492428183556
Epoch 66/100, Loss: 1.8415989056229591
Epoch 67/100, Loss: 1.8177480027079582
Epoch 68/100, Loss: 1.7884052023291588
Epoch 69/100, Loss: 1.886004276573658
Epoch 70/100, Loss: 1.858305163681507
Epoch 71/100, Loss: 1.761992760002613
Epoch 72/100, Loss: 1.8299240805208683
Epoch 73/100, Loss: 1.7891704328358173


Epoch 74/100, Loss: 1.6550149098038673
Epoch 75/100, Loss: 1.6084844693541527
Epoch 76/100, Loss: 1.7850634008646011
Epoch 77/100, Loss: 1.7884960174560547
Epoch 78/100, Loss: 1.6189666017889977
Epoch 79/100, Loss: 2.054045271128416
Epoch 80/100, Loss: 1.7875980995595455
Epoch 81/100, Loss: 1.9193192422389984
Epoch 82/100, Loss: 1.7574251145124435
Epoch 83/100, Loss: 1.8108321577310562
Epoch 84/100, Loss: 1.7325320318341255
Epoch 85/100, Loss: 1.986321747303009
Epoch 86/100, Loss: 1.7822225391864777
Epoch 87/100, Loss: 1.887823037803173
Epoch 88/100, Loss: 1.7981762327253819
Epoch 89/100, Loss: 1.8455718234181404
Epoch 90/100, Loss: 1.7540008053183556
Epoch 91/100, Loss: 1.8684296160936356


...

Epoch 10/100, Loss: 1.5047696009278297
Epoch 11/100, Loss: 1.5497945994138718
Epoch 12/100, Loss: 1.5210253819823265
Epoch 13/100, Loss: 1.4766047596931458
Epoch 14/100, Loss: 1.5537107288837433
Epoch 15/100, Loss: 1.5642765387892723
Epoch 16/100, Loss: 1.499333769083023
Epoch 17/100, Loss: 1.5042614042758942
Epoch 18/100, Loss: 1.604661650955677
Epoch 19/100, Loss: 1.544258013367653
Epoch 20/100, Loss: 1.515917792916298
Epoch 21/100, Loss: 1.5618745610117912
Epoch 22/100, Loss: 1.478341706097126
Epoch 23/100, Loss: 1.5419260188937187
Epoch 24/100, Loss: 1.415182150900364
Epoch 25/100, Loss: 1.5431738942861557
Epoch 26/100, Loss: 1.4797919020056725
Epoch 27/100, Loss: 1.5730926916003227
Epoch 28/100, Loss: 1.52423395216465


Epoch 29/100, Loss: 1.4544041529297829
Epoch 30/100, Loss: 1.5175209641456604
Epoch 31/100, Loss: 1.5060740858316422
Epoch 32/100, Loss: 1.4487155377864838
Epoch 33/100, Loss: 1.3962467014789581
Epoch 34/100, Loss: 1.4406951069831848
Epoch 35/100, Loss: 1.5776900947093964
Epoch 36/100, Loss: 1.4725606814026833
Epoch 37/100, Loss: 1.4602159038186073
Epoch 38/100, Loss: 1.5428539738059044
Epoch 39/100, Loss: 1.5365237221121788
Epoch 40/100, Loss: 1.5186742469668388
Epoch 41/100, Loss: 1.5635004863142967
Epoch 42/100, Loss: 1.5796352475881577
Epoch 43/100, Loss: 1.665969267487526
Epoch 44/100, Loss: 1.4277668446302414
Epoch 45/100, Loss: 1.5317831486463547
Epoch 46/100, Loss: 1.6520654559135437
Epoch 47/100, Loss: 1.6253500282764435


Epoch 48/100, Loss: 1.629794456064701
Epoch 49/100, Loss: 1.5787662118673325
Epoch 50/100, Loss: 1.5713825970888138
Epoch 51/100, Loss: 1.5561320781707764
Epoch 52/100, Loss: 1.547835573554039
Epoch 53/100, Loss: 1.52768025547266
Epoch 54/100, Loss: 1.5264136865735054
Epoch 55/100, Loss: 1.6449898481369019
Epoch 56/100, Loss: 1.5066108256578445
Epoch 57/100, Loss: 1.4858613908290863
Epoch 58/100, Loss: 1.4961174950003624
Epoch 59/100, Loss: 1.6234946623444557
Epoch 60/100, Loss: 1.5240928679704666
Epoch 61/100, Loss: 1.5256245285272598
Epoch 62/100, Loss: 1.4711915701627731
Epoch 63/100, Loss: 1.5869898423552513
Epoch 64/100, Loss: 1.4344416707754135
Epoch 65/100, Loss: 1.4353443384170532


Epoch 66/100, Loss: 1.524223156273365
Epoch 67/100, Loss: 1.6616937518119812
Epoch 68/100, Loss: 1.5272523611783981
Epoch 69/100, Loss: 1.6111400052905083
Epoch 70/100, Loss: 1.5047585219144821
Epoch 71/100, Loss: 1.4869650453329086
Epoch 72/100, Loss: 1.502409189939499
Epoch 73/100, Loss: 1.5538410395383835
Epoch 74/100, Loss: 1.5819921270012856
Epoch 75/100, Loss: 1.5466314256191254
Epoch 76/100, Loss: 1.5760621801018715
Epoch 77/100, Loss: 1.4901115074753761
Epoch 78/100, Loss: 1.5274161398410797
Epoch 79/100, Loss: 1.6149084940552711
Epoch 80/100, Loss: 1.4685022234916687
Epoch 81/100, Loss: 1.5374886617064476
Epoch 82/100, Loss: 1.473382592201233
Epoch 83/100, Loss: 1.5360546931624413


Epoch 84/100, Loss: 1.5526961833238602
Epoch 85/100, Loss: 1.5706058889627457
Epoch 86/100, Loss: 1.4604935869574547
Epoch 87/100, Loss: 1.4524644613265991
Epoch 88/100, Loss: 1.5053951814770699
Epoch 89/100, Loss: 1.6329554915428162
Epoch 90/100, Loss: 1.487163431942463
Epoch 91/100, Loss: 1.4431887790560722
Epoch 92/100, Loss: 1.4579074531793594
Epoch 93/100, Loss: 1.487896852195263
Epoch 94/100, Loss: 1.582346886396408
Epoch 95/100, Loss: 1.5532917007803917
Epoch 96/100, Loss: 1.5069417282938957
Epoch 97/100, Loss: 1.5210383161902428
Epoch 98/100, Loss: 1.5717246383428574
Epoch 99/100, Loss: 1.482140228152275
Epoch 100/100, Loss: 1.5326082557439804
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.4523
